# Tutorial 5: Advanced Agent Techniques and Real-World Applications

In this tutorial, we'll explore advanced agent techniques in LangChain and apply them to create a sophisticated AI-powered research assistant.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import Tool, tool
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# llama-3.1-8b-instant: 500K tokens/day on Groq's free tier vs 100K for
# llama-3.3-70b-versatile. Agent tutorials are the most token-hungry in this series,
# so the larger daily budget is worth more than the 70b's tool-routing edge.
llm = ChatGroq(model_name='llama-3.1-8b-instant')

# Use OllamaEmbeddings if available, otherwise FakeEmbeddings
try:
    embeddings = OllamaEmbeddings(model='all-minilm', base_url=os.getenv('OLLAMA_EMBEDDING_URL'))
    _ = embeddings.embed_query('test')  # probe connection
except Exception:
    from langchain_core.embeddings import FakeEmbeddings
    embeddings = FakeEmbeddings(size=384)
    print('Ollama not available - using FakeEmbeddings for demo')

print("Setup complete.")

## 1. Creating a Custom Agent with Specialized Capabilities

In [ ]:
# create_agent (from the langchain package) handles the ReAct template and output
# parsing internally - no need for StringPromptTemplate or AgentOutputParser.
#
# The system_prompt parameter lets you customise agent behaviour. Refer to tools by
# their registered names, which for @tool functions are the function names below.
system_prompt = (
    "You are an AI research assistant for scientific literature analysis.\n"
    "When answering:\n"
    "1. Search for relevant papers using vector_store_search\n"
    "2. Summarize key findings using summarize\n"
    "3. Provide analytical insights using analyze\n"
    "Be thorough and cite specific findings from the papers."
)

print("System prompt defined.")

## 2. Implementing a Multi-Agent System

In [ ]:
# Load the research papers.
#
# DirectoryLoader/TextLoader would do this too, but they live in langchain_community,
# which was sunset on 2026-05-22 and archived read-only on 2026-06-19. For plain .txt
# files a "loader" is just a read plus a Document wrapper, so we do it directly and
# keep this tutorial free of the unmaintained package. Tutorial 3 keeps the loaders,
# where document loading is the actual subject and PyPDFLoader does real work.
documents = [
    Document(page_content=p.read_text(encoding='utf-8'), metadata={'source': str(p)})
    for p in sorted(Path('research_papers').glob('*.txt'))
]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

# InMemoryVectorStore (langchain_core) - the three sample papers here are only a
# few KB, so exact brute-force cosine similarity is both accurate and instant, with
# no extra native dependency. See Tutorial 12 for FAISS, which adds approximate
# nearest-neighbor indexing once a corpus outgrows brute-force search.
vectorstore = InMemoryVectorStore.from_documents(texts, embeddings)
print(f'Loaded {len(documents)} papers -> {len(texts)} document chunks')

# Define tools using @tool decorator (modern approach)
@tool
def vector_store_search(query: str) -> str:
    """Search research papers for relevant information."""
    results = vectorstore.similarity_search(query, k=2)
    return '\n'.join(d.page_content[:200] for d in results)

@tool
def summarize(text: str) -> str:
    """Summarize a given text concisely."""
    return llm.invoke([HumanMessage(content=f'Summarize in 3 sentences: {text[:500]}')]).content

@tool
def analyze(text: str) -> str:
    """Analyze text and provide key insights."""
    return llm.invoke([HumanMessage(content=f'Give 3 key insights from: {text[:500]}')]).content

tools = [vector_store_search, summarize, analyze]

# create_agent replaces the deprecated LLMSingleActionAgent + AgentExecutor, and
# supersedes langgraph.prebuilt.create_react_agent (also now deprecated)
research_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# Test
result = research_agent.invoke({
    'messages': [HumanMessage(content='What renewable energy technologies are discussed? Summarize the key findings.')]
})
print(result["messages"][-1].content[:400])

## 3. Developing a Context-Aware Agent with Long-Term Memory

In [ ]:
# Memory in modern LangChain: MemorySaver checkpointer on the LangGraph agent
# This replaces ConversationBufferMemory + ConversationChain

memory_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=MemorySaver()
)

config = {'configurable': {'thread_id': 'research-session-1'}}

# Turn 1: ask about climate change
r1 = memory_agent.invoke(
    {'messages': [HumanMessage(content='What methods were used in the climate change study?')]},
    config
)
print('Turn 1:', r1['messages'][-1].content[:200])

# Turn 2: follow-up — agent remembers the context
r2 = memory_agent.invoke(
    {'messages': [HumanMessage(content='What were the main findings of that study?')]},
    config
)
print("Turn 2:", r2["messages"][-1].content[:200])

## 4. Building a Real-World Application: AI-Powered Research Assistant

In [5]:
def research_assistant(query: str, cfg: dict) -> str:
    response = memory_agent.invoke(
        {'messages': [HumanMessage(content=query)]}, cfg
    )
    answer = response['messages'][-1].content
    print(f'Q: {query}')
    print(f'A: {answer[:200]}\n')
    return answer

research_config = {'configurable': {'thread_id': 'research-queries'}}

research_assistant('What are the findings about coral reefs?', research_config)
research_assistant("What AI applications in medical imaging are mentioned?", research_config)

Q: What are the findings about coral reefs?
A: The search results retrieved information about artificial intelligence in healthcare, which appears unrelated to coral reef findings. This may indicate a mismatch in the search process. Would you like

Q: What AI applications in medical imaging are mentioned?
A: The tool response you referenced contains incomplete text (cut off mid-sentence) and appears to be a placeholder or error, as it mentions "artificial intelligence in healthcare" but lacks specific det



'The tool response you referenced contains incomplete text (cut off mid-sentence) and appears to be a placeholder or error, as it mentions "artificial intelligence in healthcare" but lacks specific details about **medical imaging**. The provided snippet does not include the actual findings or applications discussed in the paper. \n\nTo address your question accurately, I would need access to the full paper or a proper summary of its contents. If you\'d like, I can:\n1. Refine the search for **AI applications in medical imaging** using updated terms.\n2. Explore **coral reef studies** further if you\'d prefer to return to that topic.\n\nLet me know how you\'d like to proceed!'

## Summary and Key Takeaways

In this tutorial, we've explored advanced agent techniques in LangChain and applied them to create a sophisticated AI-powered research assistant:

1. **Custom Agent Creation**: We developed a specialized agent for scientific literature analysis, demonstrating how to tailor agents for specific domains.

2. **Multi-Agent System**: By combining multiple tools (search, summarize, analyze), we created a versatile system capable of handling complex research tasks.

3. **Context-Aware Agent with Long-Term Memory**: We implemented a conversation memory, allowing the agent to maintain context across multiple interactions and provide more coherent and relevant responses over time.

4. **Real-World Application**: We built an AI-powered research assistant that can analyze scientific papers, extract key information, and provide insights on complex topics like climate change.

Key Takeaways:
- Advanced agents can be tailored for specific domains and tasks, greatly enhancing their effectiveness.
- Combining multiple tools and agent types allows for the creation of powerful, multi-functional AI systems.
- Implementing memory and context awareness significantly improves the quality and coherence of agent responses over time.
- Real-world applications of these techniques can lead to powerful AI assistants capable of handling complex, domain-specific tasks.

Next Steps:
- Experiment with different combinations of tools and agent types for your specific use cases.
- Explore ways to further enhance the agent's memory and context understanding capabilities.
- Consider integrating external APIs or databases to expand the agent's knowledge and capabilities.
